# Solution 4c — Wikipedia-only fine-tuning: the clean test

Solution 4b showed the gain lives in the **pilot** subset:

| subset | R@1 gain | R@5 gain | positive seeds (R@5) |
|---|---|---|---|
| pilot | +0.138 | +0.110 | 5/5 |
| wikipedia | +0.034 | −0.002 | **1/5** |

The passage-level split stopped the model memorising individual passages — but not the **author's writing style**, shared across all 80 pilot passages.

This removes pilot data entirely. Retrieval still searches all 3,054 passages (the task stays realistic), but every training pair and test question is Wikipedia-grounded. Also sweeps epochs, since 200 items is a small training set.

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **GPU required.** 15 runs, checkpointed.

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers requests accelerate

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,

    "seeds": [42, 7, 123, 2024, 777],
    "epoch_settings": [2, 3, 5],   # small data -> training length matters
    "test_frac": 0.35,
    "batch_size": 16,
    "lr": 2e-5,
    "warmup_frac": 0.1,

    "k_values": (1, 5),
    "max_k": 10,
    "bootstrap_n": 1000,
    "ci": 95,
    "checkpoint": "wiki_only_checkpoint.json",
}

### Load; Wikipedia questions only

In [ ]:
import json, random, re, os, gc
import numpy as np
import pandas as pd

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)

qa = [q for q in wiki_qa if q["source_chunk_id"] in known]

# Retrieval still searches the FULL corpus — only the questions and training
# pairs are restricted. Otherwise the task would become artificially easy.
print(f"Retrieval corpus: {len(corpus)} passages (full)")
print(f"Questions:        {len(qa)} (Wikipedia-grounded only; no pilot data)")
print(f"Gold passages:    {len({q['source_chunk_id'] for q in qa})}")

### BM25 + Arabic normalization

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t)
    t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t)
    t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_cache = {}
def bm25_scores(q):
    if q not in _cache:
        _cache[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _cache[q]

print("BM25 ready.")

### Split / train / evaluate

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch

def make_split(seed):
    rnd = random.Random(seed)
    passages = sorted({q["source_chunk_id"] for q in qa})
    rnd.shuffle(passages)
    n_test = int(len(passages) * CONFIG["test_frac"])
    test_p = set(passages[:n_test])
    train = [q for q in qa if q["source_chunk_id"] not in test_p]
    test = [q for q in qa if q["source_chunk_id"] in test_p]
    assert not ({q["source_chunk_id"] for q in train} & test_p), "passage leakage"
    return train, test

def make_examples(train_qa, seed):
    ex = []
    for q in train_qa:
        ex.append(InputExample(texts=[f'query: {q["darija_query"]}',
                                      f'passage: {corpus_map[q["source_chunk_id"]]}']))
        ex.append(InputExample(texts=[f'query: {q["darija_query"]}',
                                      f'query: {q["msa_query"]}']))
    random.Random(seed).shuffle(ex)
    return ex

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def build_index(model):
    emb = model.encode([f"passage: {t}" for t in corpus_texts],
                       normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    return np.asarray(emb, "float32")

def evaluate(model, emb, items, field):
    a = CONFIG["alpha"]
    hits = {k: [] for k in CONFIG["k_values"]}
    rr = []
    qs = [it[field] for it in items]
    qe = model.encode([f"query: {q}" for q in qs],
                      normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    for i, it in enumerate(items):
        s = a * minmax(emb @ qe[i]) + (1 - a) * minmax(bm25_scores(qs[i]))
        got = [corpus_ids[j] for j in np.argsort(-s)[:CONFIG["max_k"]]]
        gold = it["source_chunk_id"]
        for k in CONFIG["k_values"]:
            hits[k].append(1.0 if gold in got[:k] else 0.0)
        rr.append(1.0 / (got.index(gold) + 1) if gold in got else 0.0)
    return {**{f"R@{k}": np.array(v) for k, v in hits.items()}, "MRR": np.array(rr)}

def finetune(examples, epochs):
    model = SentenceTransformer(CONFIG["base_encoder"])
    loader = DataLoader(examples, shuffle=True, batch_size=CONFIG["batch_size"])
    loss = losses.MultipleNegativesRankingLoss(model)
    steps = len(loader) * epochs
    model.fit(train_objectives=[(loader, loss)], epochs=epochs,
              warmup_steps=int(steps * CONFIG["warmup_frac"]),
              optimizer_params={"lr": CONFIG["lr"]}, show_progress_bar=False)
    return model

### Run (checkpointed)

In [ ]:
records = []
if os.path.exists(CONFIG["checkpoint"]):
    records = json.load(open(CONFIG["checkpoint"], encoding="utf-8"))
    print(f"Resuming with {len(records)} records.")
done = {(r["seed"], r["epochs"]) for r in records}
total = len(CONFIG["seeds"]) * len(CONFIG["epoch_settings"])

# Per-item vectors for the best setting are kept for bootstrapping later.
per_item_store = {}

for seed in CONFIG["seeds"]:
    train_qa, test_qa = make_split(seed)

    base = SentenceTransformer(CONFIG["base_encoder"])
    base_emb = build_index(base)
    before = {f: evaluate(base, base_emb, test_qa, f) for f in ["msa_query", "darija_query"]}
    del base, base_emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    for epochs in CONFIG["epoch_settings"]:
        if (seed, epochs) in done:
            continue
        print(f"\n[{len(records)+1}/{total}] seed={seed} epochs={epochs} "
              f"(train {len(train_qa)} / test {len(test_qa)})")

        model = finetune(make_examples(train_qa, seed), epochs)
        emb = build_index(model)
        after = {f: evaluate(model, emb, test_qa, f) for f in ["msa_query", "darija_query"]}

        rec = {"seed": seed, "epochs": epochs,
               "n_train": len(train_qa), "n_test": len(test_qa)}
        for m in ["R@1", "R@5", "MRR"]:
            rec[f"darija_before_{m}"] = float(before["darija_query"][m].mean())
            rec[f"darija_after_{m}"]  = float(after["darija_query"][m].mean())
            rec[f"msa_before_{m}"]    = float(before["msa_query"][m].mean())
            rec[f"msa_after_{m}"]     = float(after["msa_query"][m].mean())
        records.append(rec)
        json.dump(records, open(CONFIG["checkpoint"], "w", encoding="utf-8"), indent=2)

        per_item_store[(seed, epochs)] = {
            "before_darija": before["darija_query"]["R@5"].tolist(),
            "after_darija":  after["darija_query"]["R@5"].tolist(),
        }

        del model, emb
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df = pd.DataFrame(records)
df.to_csv("wiki_only_raw.csv", index=False)
print(f"\nComplete: {len(df)} runs -> wiki_only_raw.csv")

### THE VERDICT: does fine-tuning help on Wikipedia-only data?

In [ ]:
print("=" * 86)
print("WIKIPEDIA-ONLY FINE-TUNING — no pilot data anywhere in the pipeline")
print("=" * 86)

for epochs in CONFIG["epoch_settings"]:
    s = df[df.epochs == epochs]
    if s.empty:
        continue
    print(f"\n--- {epochs} epochs (n={len(s)} seeds) ---")
    for m in ["R@1", "R@5", "MRR"]:
        g = s[f"darija_after_{m}"] - s[f"darija_before_{m}"]
        msa = s[f"msa_after_{m}"] - s[f"msa_before_{m}"]
        print(f"  {m:<5} Darija {s[f'darija_before_{m}'].mean():.3f} -> "
              f"{s[f'darija_after_{m}'].mean():.3f}  "
              f"gain {g.mean():+.4f} (sd {g.std():.4f}, positive {(g>0).sum()}/{len(g)})   "
              f"MSA {msa.mean():+.4f}")

print("\nA gain positive in 4-5 of 5 seeds, at more than one epoch setting,")
print("is a real effect on real text. A gain positive in 1-2 seeds is not.")

### Compare directly against the mixed-data result

In [ ]:
print("\n" + "=" * 86)
print("COMPARISON — what the pilot data was contributing")
print("=" * 86)
best_ep = None
best_gain = -1
for epochs in CONFIG["epoch_settings"]:
    s = df[df.epochs == epochs]
    if s.empty:
        continue
    g = (s["darija_after_R@5"] - s["darija_before_R@5"]).mean()
    if g > best_gain:
        best_gain, best_ep = g, epochs

print(f"""
                              R@1 gain    R@5 gain
  mixed data (Solution 4b)     +0.094      +0.064     <- inflated by pilot
    of which pilot subset      +0.138      +0.110
    of which wikipedia         +0.034      -0.002
  wikipedia-only (this run)    {(df[df.epochs==best_ep]['darija_after_R@1'] - df[df.epochs==best_ep]['darija_before_R@1']).mean():+.3f}      {best_gain:+.3f}     <- best epoch setting = {best_ep}
""")
print("If the wikipedia-only gain is clearly positive, training on pilot data was")
print("diluting a real effect. If it is near zero, the mixed-data result was an")
print("artifact of shared authorship and the paper must say so.")

### Gap before/after at the best setting

In [ ]:
print("\n" + "=" * 86)
print(f"DIALECT GAP at {best_ep} epochs")
print("=" * 86)
s = df[df.epochs == best_ep]
for m in ["R@1", "R@5", "MRR"]:
    gb = (s[f"msa_before_{m}"] - s[f"darija_before_{m}"]).mean()
    ga = (s[f"msa_after_{m}"] - s[f"darija_after_{m}"]).mean()
    print(f"  {m:<5} gap {gb:.3f} -> {ga:.3f}  ({(gb-ga)/gb*100 if gb else 0:+.1f}% of the gap closed)")

df.to_csv("wiki_only_results.csv", index=False)
from google.colab import files
files.download("wiki_only_raw.csv")